[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/PythonForDataAnalysis/blob/master/pandas%E7%9A%84%E6%AA%94%E6%A1%88%E5%AD%98%E5%8F%96/%E5%AD%B8%E7%94%9F%E7%B4%A0%E6%9D%90/%E7%AF%84%E4%BE%8B_%E8%8C%B6%E8%AA%9E%E6%99%82%E5%85%89_%E4%BA%94%E7%A8%AE%E9%9B%A3%E6%90%9E%E7%9A%84%E8%B3%87%E6%96%99%E6%AA%94.ipynb)

# 📥 茶語時光：搞定五種難搞的資料檔

> 🧋 **情境**：月初結帳日，各部門丟來五種格式的檔案：POS 系統匯出的「髒」CSV、每家店一個工作表的 Excel、外送平台的巢狀 JSON、IT 部門的會員資料庫，還有一個 10 萬筆、用 Excel 打開會卡住的交易檔。你要用 pandas 一一讀進來，最後整理成一份多工作表的 Excel 交給主管。

| 素材檔案 | 內容 |
| --- | --- |
| `POS匯出_信義店_2024-03.csv` | POS 匯出的日報：有註解行、千分位逗號、`--` 缺值、最後一列是合計 |
| `各店月報_2024.xlsx` | 5 個工作表（每店一張），前兩列是標題 |
| `外送平台訂單_2024-03.json` | 外送平台的巢狀 JSON：一筆訂單裡有多個品項 |
| `會員資料庫.db` | SQLite 資料庫，內有「會員」資料表（800 位） |
| `全年交易紀錄_2024.csv` | 10 萬筆交易紀錄（用來練習分批讀取） |

> ⚠️ 「茶語時光」是課程虛構的品牌，資料為模擬資料；節日與颱風假的日期參考台灣實際情況，方便你對照生活經驗。

In [1]:
# 📦 準備素材：在 Colab 開啟時，自動下載本章素材壓縮檔並解壓縮
#    （若是在自己電腦解壓縮後開啟，檔案已經在旁邊，會直接使用）
import os, zipfile, urllib.request

FILES = [
    'POS匯出_信義店_2024-03.csv',
    '各店月報_2024.xlsx',
    '外送平台訂單_2024-03.json',
    '會員資料庫.db',
    '全年交易紀錄_2024.csv',
]
ZIP_URL = 'https://github.com/roberthsu2003/PythonForDataAnalysis/raw/master/pandas%E7%9A%84%E6%AA%94%E6%A1%88%E5%AD%98%E5%8F%96/pandas%E7%9A%84%E6%AA%94%E6%A1%88%E5%AD%98%E5%8F%96_%E5%AD%B8%E7%94%9F%E7%B4%A0%E6%9D%90.zip'

if not all(os.path.exists(f) for f in FILES):
    urllib.request.urlretrieve(ZIP_URL, 'materials.zip')            # 下載本章素材壓縮檔
    with zipfile.ZipFile('materials.zip') as z:
        for name in z.namelist():
            inner = name.split('/', 1)[1] if '/' in name else name   # 去掉壓縮檔最外層的資料夾名稱
            if inner and not name.endswith('/'):
                os.makedirs(os.path.dirname(inner) or '.', exist_ok=True)
                with z.open(name) as src, open(inner, 'wb') as dst:
                    dst.write(src.read())

print('✅ 素材準備完成：', [f for f in FILES if os.path.exists(f)])

✅ 素材準備完成： ['POS匯出_信義店_2024-03.csv', '各店月報_2024.xlsx', '外送平台訂單_2024-03.json', '會員資料庫.db', '全年交易紀錄_2024.csv']


## 任務 1：POS 匯出的「髒」CSV——讀檔時就順便清乾淨

**先用文字方式看看原始檔長什麼樣子**：

In [2]:
print(open('POS匯出_信義店_2024-03.csv', encoding='utf-8-sig').read()[:600])

# 茶語時光 POS 系統匯出報表
# 門市：台北信義店　期間：2024-03-01 ~ 2024-03-31
# 匯出時間：2024-04-01 09:12:33
日期,來客數,杯數,營收,現金,電子支付,備註
2024-03-01,"273","399","22,125","5,863","16,262",
2024-03-02,"155","238","13,625","3,950","9,675",
2024-03-03,"151","236","13,610","5,371","8,239",
2024-03-04,"253","384","22,252","8,368","13,884",
2024-03-05,"208","307","16,801","6,466","10,335",
2024-03-06,"193","292","15,966","4,663","11,303",
2024-03-07,"240","395","21,426","7,020","14,406",
2024-03-08,--,--,--,--,--,系統當機未記錄
2024-03-09,"183","290","16,863","6,156","10,707",
2024-03-10,"135","208","11,034","3,995","7,039",
2024-03-11,"


**問題**：前 3 行是說明、數字有千分位逗號、`--` 代表當機沒記錄、最後一列是「合計」。直接讀會一團亂，我們用參數一次處理：

In [3]:
import pandas as pd

pos = pd.read_csv('POS匯出_信義店_2024-03.csv',
                  encoding='utf-8-sig',     # 檔案開頭有 BOM，用 utf-8-sig 讀
                  comment='#',              # # 開頭的說明行略過
                  thousands=',',            # "28,560" → 28560
                  na_values=['--'],         # -- 視為缺值
                  skipfooter=1,             # 最後一列「合計」不要
                  engine='python',          # 使用 skipfooter 需要 python 引擎
                  parse_dates=['日期'])      # 日期欄轉成時間型態
print(pos.dtypes)
print('\n缺資料的日子：\n', pos[pos['營收'].isnull()][['日期', '備註']])
print('\n3 月營收合計（不含當機日）：', f"{pos['營收'].sum():,.0f}")

日期      datetime64[ns]
來客數            float64
杯數             float64
營收             float64
現金             float64
電子支付           float64
備註              object
dtype: object

缺資料的日子：
            日期       備註
7  2024-03-08  系統當機未記錄
18 2024-03-19  系統當機未記錄

3 月營收合計（不含當機日）： 551,832


> 🤖 **換你問 AI**：「這是 CSV 的前 8 行（貼上），請用 pandas read_csv 讀取，跳過說明行、處理千分位、把 -- 當成缺值、不要讀最後的合計列。」

## 任務 2：每家店一個工作表的 Excel——一次讀完再合併

**白話說明**：`sheet_name=None` 會一次讀進「所有工作表」，得到一個「工作表名稱 → 表格」的字典；`header=2` 表示第 3 列才是欄位名稱。

In [4]:
sheets = pd.read_excel('各店月報_2024.xlsx', sheet_name=None, header=2)   # 所有工作表、第 3 列是標題
print('工作表：', list(sheets.keys()))

yearly = pd.concat(sheets, names=['門市', '列'])                            # 把 5 張表上下合併，工作表名稱變成「門市」欄
yearly = yearly.reset_index(level='門市').reset_index(drop=True)
print(yearly.head())

summary = yearly.pivot_table(index='月份', columns='門市', values='營收', sort=False)   # 月份 × 門市 的營收總表
print((summary / 10000).round(1))

工作表： ['台北信義店', '台北西門店', '台中逢甲店', '台南成大店', '高雄巨蛋店']
      門市  月份      營收     杯數
0  台北信義店  1月  501875   8886
1  台北信義店  2月  478108   8553
2  台北信義店  3月  590278  10507
3  台北信義店  4月  639347  11313
4  台北信義店  5月  749307  13030
門市   台北信義店  台北西門店  台中逢甲店  台南成大店  高雄巨蛋店
月份                                    
1月    50.2   55.7   68.1   49.6   57.7
2月    47.8   53.0   64.0   47.0   54.0
3月    59.0   67.9   77.7   53.1   65.7
4月    63.9   69.3   79.0   59.1   61.8
5月    74.9   81.4   89.4   60.3   70.9
6月    80.0   92.8   99.0   68.9   82.5
7月    78.9   86.6   91.9   63.9   74.3
8月    80.7   92.7   99.7   68.0   76.7
9月    71.3   80.7   87.8   61.9   75.4
10月   66.6   72.0   81.1   55.7   61.8
11月   57.7   65.5   79.0   55.0   64.2
12月   56.9   62.0   74.6   53.8   65.2


## 任務 3：外送平台的巢狀 JSON——攤平成表格

**白話說明**：一筆訂單裡有好幾個品項（清單包清單）。`pd.json_normalize` 可以「攤平」：每個品項一列，並把訂單編號、門市等外層資訊帶下來。

In [5]:
import json

with open('外送平台訂單_2024-03.json', encoding='utf-8') as f:
    raw = json.load(f)
print('平台：', raw['平台'], '｜抽成：', raw['抽成比例'], '｜訂單數：', len(raw['訂單']))

items = pd.json_normalize(raw['訂單'],
                          record_path='品項',                                  # 要攤開的清單
                          meta=['訂單編號', '下單時間', ['門市', '名稱'], '外送員評分'])   # 一起帶下來的外層欄位
items['小計'] = items['數量'] * items['單價']
print(items.head())

by_store = items.groupby('門市.名稱')['小計'].sum()
print('\n各店外送營收：\n', by_store)
print('\n扣掉 30% 平台抽成，實拿：', f"{by_store.sum() * (1 - raw['抽成比例']):,.0f}", '元')

平台： FoodRun 外送 ｜抽成： 0.3 ｜訂單數： 360
       品項  數量  單價     訂單編號              下單時間  門市.名稱 外送員評分   小計
0     多多綠   3  50  FR00001  2024-03-28 12:37  台北西門店     5  150
1   四季春青茶   1  35  FR00001  2024-03-28 12:37  台北西門店     5   35
2   四季春青茶   3  35  FR00002  2024-03-22 10:03  台北西門店     4  105
3  黑糖珍珠鮮奶   1  75  FR00002  2024-03-22 10:03  台北西門店     4   75
4  黑糖珍珠鮮奶   3  75  FR00002  2024-03-22 10:03  台北西門店     4  225

各店外送營收：
 門市.名稱
台中逢甲店    17675
台北信義店    18525
台北西門店    19010
高雄巨蛋店    21915
Name: 小計, dtype: int64

扣掉 30% 平台抽成，實拿： 53,988 元


> 💡 **看懂結果**：外送營收看起來不少，但平台要抽 30%——這就是為什麼很多飲料店的外送價格比店內貴。
>
> 🤖 **換你問 AI**：「這是外送平台 JSON 的前 30 行（貼上），請用 json_normalize 把每個品項攤成一列，並帶上訂單編號與門市名稱。」

## 任務 4：IT 部門的會員資料庫——用 SQL 撈資料

In [6]:
import sqlite3

con = sqlite3.connect('會員資料庫.db')                                       # 連線到資料庫檔案
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con))  # 資料庫裡有哪些資料表

new_2024 = pd.read_sql("""
    SELECT 常去門市, COUNT(*) AS 新會員數, ROUND(AVG(累積點數), 1) AS 平均點數
    FROM 會員
    WHERE 加入日期 >= '2024-01-01'
    GROUP BY 常去門市
    ORDER BY 新會員數 DESC
""", con)                                                                      # 2024 年加入的新會員，依門市統計
con.close()
print(new_2024)

  name
0   會員
    常去門市  新會員數   平均點數
0  台北西門店    71  300.9
1  台南成大店    66  284.7
2  高雄巨蛋店    63  253.4
3  台中逢甲店    54  278.8
4  台北信義店    47  319.1


## 任務 5：10 萬筆大檔——分批讀取不卡頓

In [7]:
print(pd.read_csv('全年交易紀錄_2024.csv', nrows=5))                         # 先偷看前 5 筆

total = pd.Series(dtype=float)
for chunk in pd.read_csv('全年交易紀錄_2024.csv', usecols=['付款方式', '金額'], chunksize=20_000):   # 每次只讀 2 萬筆、2 個欄位
    total = total.add(chunk.groupby('付款方式')['金額'].sum(), fill_value=0)      # 逐批累加
share = (total / total.sum() * 100).round(1).sort_values(ascending=False)
print('\n各付款方式金額占比 %：\n', share)

       交易編號                 交易時間     門市   金額      付款方式
0  X0000001  2024-01-01 10:07:23  台南成大店   57      街口支付
1  X0000002  2024-01-01 10:07:40  台南成大店   91  LINE Pay
2  X0000003  2024-01-01 10:09:34  高雄巨蛋店   74       悠遊卡
3  X0000004  2024-01-01 10:11:51  台北信義店  300       悠遊卡
4  X0000005  2024-01-01 10:15:59  台南成大店  372       信用卡



各付款方式金額占比 %：
 付款方式
現金           32.0
LINE Pay     21.8
悠遊卡          14.0
信用卡          12.1
Apple Pay    10.1
街口支付         10.0
dtype: float64


> 💡 **看懂結果**：電子支付（LINE Pay、悠遊卡、街口、信用卡、Apple Pay）加起來已經超過現金——櫃台是不是該加裝感應設備、推出電子支付優惠？

## 任務 6：整理成一份多工作表 Excel 交給主管

In [8]:
with pd.ExcelWriter('三月結帳報告.xlsx') as writer:                           # 一個 Excel 檔、多個工作表
    pos.to_excel(writer, sheet_name='信義店3月日報', index=False)
    (summary / 10000).round(1).to_excel(writer, sheet_name='各店月營收(萬)')
    by_store.to_frame('外送營收').to_excel(writer, sheet_name='外送營收')
    new_2024.to_excel(writer, sheet_name='2024新會員', index=False)
    share.to_frame('占比%').to_excel(writer, sheet_name='付款方式')
print('✅ 已輸出 三月結帳報告.xlsx：', list(pd.read_excel('三月結帳報告.xlsx', sheet_name=None).keys()))

items.to_csv('外送品項明細.csv', index=False, encoding='utf-8-sig')            # utf-8-sig：用 Excel 開也不會亂碼

✅ 已輸出 三月結帳報告.xlsx： ['信義店3月日報', '各店月營收(萬)', '外送營收', '2024新會員', '付款方式']


---

## ✍️ 練習題

1. 從 `各店月報_2024.xlsx` 只讀取「台南成大店」這一個工作表，找出杯數最多的月份。
2. 用 SQL 從資料庫撈出「累積點數前 10 名」的會員姓名與常去門市。
3. 分批讀取交易紀錄，計算「每家門市」的交易筆數。

## 🔑 參考解答（先自己試試看！）

In [9]:
# 練習 1
tn = pd.read_excel('各店月報_2024.xlsx', sheet_name='台南成大店', header=2)
print(tn.loc[tn['杯數'].idxmax()])

# 練習 2
con = sqlite3.connect('會員資料庫.db')
print(pd.read_sql('SELECT 姓名, 常去門市, 累積點數 FROM 會員 ORDER BY 累積點數 DESC LIMIT 10', con))
con.close()

# 練習 3
count = pd.Series(dtype=float)
for chunk in pd.read_csv('全年交易紀錄_2024.csv', usecols=['門市'], chunksize=20_000):
    count = count.add(chunk['門市'].value_counts(), fill_value=0)
print(count.astype(int).sort_values(ascending=False))

月份        6月
營收    689489
杯數     11799
Name: 5, dtype: object
    姓名   常去門市  累積點數
0  高心怡  高雄巨蛋店  1420
1  莊柏翰  高雄巨蛋店  1412
2  吳承恩  台南成大店  1230
3  賴俊傑  高雄巨蛋店  1203
4  徐宜庭  台北西門店  1198
5  何志明  台北西門店  1090
6  江柏翰  高雄巨蛋店  1082
7  林志明  台北信義店  1068
8  許怡君  台南成大店  1014
9  洪宇翔  台南成大店  1004
門市
台中逢甲店    23732
台北西門店    20987
台北信義店    19132
高雄巨蛋店    19079
台南成大店    17070
dtype: int64
